In [1]:

import os
import re
import argparse
import pandas as pd
from pathlib import Path
from typing import List, Tuple

# ----------------------------
# CONFIGURABLE CONSTANTS
# ----------------------------
RAD_SECTION = "&noahmp_rad_parameters"
USGS_SECTION = "&noahmp_usgs_parameters"
SOIL_SECTION = "&noahmp_soil_stas_parameters"

RAD_VARS = {"ALBSAT_VIS", "ALBSAT_NIR", "ALBDRY_VIS", "ALBDRY_NIR"}
USGS_SUFFIX_TO_TYPE = {"EBF": 13, "CP": 2, "SAV": 10}
SOIL_SUFFIX_TO_TYPE = {"CL": 9, "loam": 6, "SCL": 7}

# ----------------------------
# HELPERS
# ----------------------------
def build_var_info(columns: List[str]) -> pd.DataFrame:
    """Build the (section, variable, type) mapping for each samples column name."""
    rows = []
    for col in columns:
        if col in RAD_VARS:
            rows.append((col, RAD_SECTION, col, 4))
        elif "_" in col:
            base, suffix = col.rsplit("_", 1)
            if suffix in USGS_SUFFIX_TO_TYPE:
                rows.append((col, USGS_SECTION, base, USGS_SUFFIX_TO_TYPE[suffix]))
            elif suffix in SOIL_SUFFIX_TO_TYPE:
                rows.append((col, SOIL_SECTION, base, SOIL_SUFFIX_TO_TYPE[suffix]))
            else:
                rows.append((col, "UNKNOWN", base, None))
        else:
            rows.append((col, "UNKNOWN", col, None))

    return pd.DataFrame(rows, columns=["column_name", "section", "variable", "type"])


def find_section_span(text: str, section_name: str) -> Tuple[int, int]:
    """Return (start_idx, end_idx) of a section delimited by '&...\\n' to next section start or EOF."""
    start = text.find(section_name)
    if start == -1:
        raise ValueError(f"Section '{section_name}' not found in table.")
    # Find the next section start ('\n&') after 'start'
    next_match = re.search(r"\n&", text[start+1:])
    if next_match:
        end = start + 1 + next_match.start()
    else:
        end = len(text)
    return start, end


def replace_var_value_in_section(section_text: str, var_name: str, type_idx_1based: int, new_value: float) -> str:
    """
    Replace the comma-separated {type_idx}-th value on the var assignment line within section_text.
    Assumes assignments are single-line like: 'VAR = v1, v2, v3, ...'
    Preserves anything after an inline comment marker '!' by keeping it as-is.
    """
    # Regex for the assignment line
    pattern = re.compile(rf"(^[ \t]*{re.escape(var_name)}[ \t]*=[^\n]*$)", flags=re.MULTILINE)
    m = pattern.search(section_text)
    if not m:
        raise ValueError(f"Variable '{var_name}' not found as single-line assignment within section.")

    full_line = m.group(1)
    # Split into lhs, rhs
    lhs, rhs = full_line.split("=", 1)
    # Separate inline comment if any
    comment_part = ""
    if "!" in rhs:
        rhs, comment_part = rhs.split("!", 1)
        comment_part = "!" + comment_part  # keep marker

    # Tokenize comma-separated values
    parts = [p.strip() for p in rhs.strip().split(",")]
    idx = type_idx_1based - 1
    if not (0 <= idx < len(parts)):
        raise IndexError(f"Type index {type_idx_1based} is out of range for variable '{var_name}' with {len(parts)} values.")

    # Replace target value (format minimally)
    parts[idx] = f"{float(new_value):.8g}"

    new_rhs = ", ".join(parts)
    new_line = f"{lhs.strip()} = {new_rhs}{comment_part}".rstrip()

    # Put back original indentation from lhs
    indent = re.match(r"^[ \t]*", full_line).group(0)
    new_line = indent + new_line + "\n"

    # Replace in section_text
    start, end = m.span(1)
    section_text = section_text[:start] + new_line + section_text[end:]
    return section_text


def apply_row_to_table(base_text: str, var_info: pd.DataFrame, row: pd.Series) -> str:
    """Apply one samples row to the full table text and return updated text."""
    text = base_text  # work on a copy

    # We'll update per section to avoid shifting indices too much.
    for section_name in (RAD_SECTION, USGS_SECTION, SOIL_SECTION):
        # Slice the section span fresh each time (indices shift after replacements)
        start, end = find_section_span(text, section_name)
        section_text = text[start:end]

        # Extract the variables that live in this section, in the order they appear in var_info
        idxs = var_info.index[var_info["section"] == section_name].tolist()
        for j in idxs:
            var = var_info.loc[j, "variable"]
            typ = int(var_info.loc[j, "type"])
            col_name = var_info.loc[j, "column_name"]
            if pd.isna(row[col_name]):
                # Skip NaNs (no replacement)
                continue
            section_text = replace_var_value_in_section(section_text, var, typ, row[col_name])

        # Put the updated section back
        text = text[:start] + section_text + text[end:]
    return text


def main():
    parser = argparse.ArgumentParser(description="Apply NoahMP samples to NoahmpTable.TBL to generate emulated tables.")
    parser.add_argument("--samples", required=True, help="Path to noahmp_1000samples.txt (space-delimited).")
    parser.add_argument("--base_table", required=True, help="Path to the base NoahmpTable.TBL to modify.")
    parser.add_argument("--out_root", default=".", help="Output root directory.")
    parser.add_argument("--n_rows", type=int, default=10, help="How many rows to process (default: 10).")
    # args = parser.parse_args()
    args = parser.parse_args(['--samples', 'noahmp_1000samples.txt', '--base_table', 'NoahmpTable.TBL',
    '--out_root', './', '--n_rows', '10'])

    # Load samples
    df = pd.read_csv(args.samples, delim_whitespace=True)
    var_info = build_var_info(df.columns.tolist())

    # Sanity check: any unknowns?
    unknowns = var_info[var_info["section"] == "UNKNOWN"]
    if not unknowns.empty:
        raise RuntimeError(f"Found columns with unknown section/type:\n{unknowns}")

    # Load base table once
    with open(args.base_table, "r", encoding="utf-8") as f:
        base_text = f.read()

    n = min(args.n_rows, len(df))
    for i in range(n):
        updated_text = apply_row_to_table(base_text, var_info, df.iloc[i])

        out_dir = Path(args.out_root) / f"NoahmpTable_4emu_{i+1}"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "NoahmpTable.TBL"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(updated_text)
        print(f"Wrote {out_path}")

if __name__ == "__main__":
    main()


Wrote NoahmpTable_4emu_1\NoahmpTable.TBL
Wrote NoahmpTable_4emu_2\NoahmpTable.TBL
Wrote NoahmpTable_4emu_3\NoahmpTable.TBL
Wrote NoahmpTable_4emu_4\NoahmpTable.TBL
Wrote NoahmpTable_4emu_5\NoahmpTable.TBL
Wrote NoahmpTable_4emu_6\NoahmpTable.TBL
Wrote NoahmpTable_4emu_7\NoahmpTable.TBL
Wrote NoahmpTable_4emu_8\NoahmpTable.TBL
Wrote NoahmpTable_4emu_9\NoahmpTable.TBL
Wrote NoahmpTable_4emu_10\NoahmpTable.TBL


C:\Users\15330\AppData\Local\Temp\ipykernel_23388\3712030942.py:136: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(args.samples, delim_whitespace=True)
